# Lab 1 — Clear the queue with a crew

**~25 minutes · nothing to fill in · Run All takes 1 to 2 minutes**

Earlier in the course you built an agent as a graph. You wrote every step and every edge yourself.

This lab builds a support desk in a different way, with **CrewAI**. You describe **who** is on the team
and **what** each member does. CrewAI then decides the order of the model calls.

**The job.** Global Bank customers send in tickets. Five are waiting in the queue. For each ticket the
desk must:

1. send it to the **team that owns it**, with a priority,
2. write the **first reply** the customer reads, and
3. write a **handover note** for that team: what happened, what to check, and what to do next.

At the end you have the whole queue handled, and you check the desk's work against the right answers.

**Words used in this lab**

- **Agent:** a model with a job description. It can also call tools.
- **Tool:** a Python function that the model can ask to run, for example to look up a ticket.
- **Task:** one piece of work for one agent. It says what to do and what the answer must look like.
- **Crew:** a group of agents and the tasks they run. This is CrewAI's word.
- **Handover note:** the note the owning team reads, so it can act without asking the desk again.

## 1 · The model

Every call goes to the course's model gateway. The gateway is one address that serves several models.
Your sandbox already has three settings for it: the model name, the gateway address and your key. You
do not paste a key anywhere.

**CrewAI takes the plain model name.** You do not add an `openai/` prefix. CrewAI 1.x connects to
OpenAI-style APIs directly, so it does not need one. Google ADK, in Lab 3, is the opposite. If you copy
this cell into an ADK notebook, it fails.

In [ ]:
import os
from crewai import LLM

llm = LLM(
    model=os.environ["OPENAI_MODEL"],        # plain name - no "openai/" prefix
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)
print("model:", llm.model)
print("provider:", llm.provider)      # "openai" is the default

## 2 · The queue, the teams and a tool

`desk_kit.py`, next to this notebook, holds the five tickets and the four teams. All four labs use it,
so every lab works on the same queue. Open the file if you want to read it.

Each team has one line that says what it owns. The desk reads those lines to decide where a ticket
goes. Without them, "update my address" could go to Accounts or to Customer.

The tool below stands in for the bank's ticket system. The docstring matters: the model reads it to
decide whether to call the tool.

In [ ]:
from desk_kit import TICKETS, TEAMS, TEAM_RULE, EXPECTED_TEAM, check_reply, print_queue, score

for ticket_id, text in TICKETS.items():
    print(ticket_id, "-", text)
print()
for team, owns in TEAMS.items():
    print(f"{team:<15} owns {owns}")

from crewai.tools import tool

@tool("ticket_lookup")
def ticket_lookup(ticket_id: str) -> str:
    """Return the text of a Global Bank customer support ticket by its id, e.g. GB-T-4471."""
    return TICKETS.get(ticket_id, f"No ticket found with id {ticket_id}")

## 3 · One agent, one task

In CrewAI an **agent** is three strings: a **role**, a **goal** and a **backstory**. CrewAI turns them
into the system prompt. The system prompt is the set of instructions the model reads before the work
itself.

A **task** has a `description` and an `expected_output`. The `expected_output` is more important than
it looks. The next task in the crew relies on it, so treat it as a contract.

`allow_delegation=False` stops this agent from passing its work to another agent. Delegation is one of
the failure modes at the end of Lab 2.

> **Why `await crew.kickoff_async()` and not `crew.kickoff()`?** A notebook already runs an event loop.
> CrewAI 1.x refuses to start a synchronous run inside a running loop. It raises
> `RuntimeError: Agent execution was invoked synchronously from within a running event loop`.
> So in a notebook, use **`await crew.kickoff_async()`**. In a plain `.py` script, `crew.kickoff()` is
> correct. Most CrewAI examples on the web use `kickoff()`, so watch for this when you copy one into
> Jupyter.

In [ ]:
from crewai import Agent, Task, Crew, Process

triage = Agent(
    role="Support Triage Analyst",
    goal="Classify an incoming ticket and name the team that owns it",
    backstory="You triage the Global Bank customer support desk. You always send a ticket to the correct team.",
    llm=llm,
    tools=[ticket_lookup],
    allow_delegation=False,
    verbose=False,
)

classify = Task(
    description=f"Look up ticket GB-T-4471 and classify it. Give the category and the owning team. {TEAM_RULE}",
    expected_output="Exactly two lines - 'Category: <x>' then 'Team: <y>'",
    agent=triage,
)

crew = Crew(agents=[triage], tasks=[classify], process=Process.sequential, verbose=False)
result = await crew.kickoff_async()
print(result)

**Look at what you did not write.** You wrote no graph, no edges, no state object and no routing
function. You named a role and described a task. This is the main trade-off in CrewAI: you write less,
and the framework decides more.

The answer is two lines of text. That is fine for a person to read. A program that must send the
ticket to a team needs a **record** with named fields, not a paragraph. That comes next.

## 4 · Make each answer a record

`output_pydantic` tells CrewAI the exact shape of a task's answer. You describe the shape as a Pydantic
class. CrewAI asks the model for it and gives you back an object with named fields.

- `Triage` is the routing decision: category, team and priority.
- `DeskReply` is what the desk sends out: the customer reply, and the three parts of the handover note.

The `description` of each field is read by the model, just like a tool's docstring.

In [ ]:
from pydantic import BaseModel, Field

class Triage(BaseModel):
    category: str = Field(description="The kind of problem, in a few words")
    team: str = Field(description="The team that owns the ticket")
    priority: str = Field(description="High, Medium or Low")

class DeskReply(BaseModel):
    customer_reply: str = Field(description="The reply the customer reads, at most four sentences")
    summary: str = Field(description="Handover: what happened, in one sentence")
    what_to_check: str = Field(description="Handover: what the owning team should check first")
    next_action: str = Field(description="Handover: the next thing the owning team should do")

print(list(Triage.model_fields))
print(list(DeskReply.model_fields))

## 5 · Three agents, in order

Now a real crew with three agents. `Process.sequential` runs the tasks one after another, in list
order. Each task's output becomes context for the next task. You still do not write the wiring. The
order of the `tasks` list is the wiring.

The descriptions say `{ticket_id}`, not a real id. You fill it in when you start the crew with
`kickoff_async(inputs={"ticket_id": ...})`. So one crew can handle every ticket in the queue.

In [ ]:
researcher = Agent(
    role="Ticket Researcher",
    goal="Retrieve the ticket and state the facts in it, without interpreting them",
    backstory="You pull the raw ticket and never guess beyond what it says.",
    llm=llm, tools=[ticket_lookup], allow_delegation=False,
)

classifier = Agent(
    role="Support Triage Analyst",
    goal="Classify a ticket, name the owning team and set a priority",
    backstory="You triage the Global Bank customer support desk.",
    llm=llm, allow_delegation=False,
)

writer = Agent(
    role="Response Drafter",
    goal="Write the first reply the bank customer will read, and the handover note for the owning team",
    backstory="You write to Global Bank customers. You write plainly, promise only what the team can do, "
              "and never invent a timeline.",
    llm=llm, allow_delegation=False,
)

t1 = Task(description="Retrieve ticket {ticket_id} and list the facts it contains.",
          expected_output="A short bulleted list of facts, no interpretation.", agent=researcher)
t2 = Task(description=f"Classify ticket {{ticket_id}} and name the owning team. {TEAM_RULE} "
                      "The priority is High if the customer has lost money, otherwise Medium or Low.",
          expected_output="The category, the team and the priority.", agent=classifier,
          output_pydantic=Triage)
t3 = Task(description="Write the first reply to the customer who sent ticket {ticket_id}, "
                      "and a handover note for the owning team.",
          expected_output="The customer reply and the three parts of the handover note.", agent=writer,
          output_pydantic=DeskReply)

desk = Crew(agents=[researcher, classifier, writer], tasks=[t1, t2, t3],
            process=Process.sequential, verbose=False)

out = await desk.kickoff_async(inputs={"ticket_id": "GB-T-4471"})
decision, reply = t2.output.pydantic, out.pydantic

print("team     :", decision.team, "| priority:", decision.priority, "| category:", decision.category)
print("reply    :", reply.customer_reply)
print()
print("HANDOVER to", decision.team)
print("  what happened :", reply.summary)
print("  check first   :", reply.what_to_check)
print("  next action   :", reply.next_action)

**Read it as the Transactions team would.** Could they start work from the handover note alone,
without opening the ticket? That is the test of a good handover note.

`t2.output.pydantic` is the routing decision from the second task. `out.pydantic` is the last task's
record. Both are ordinary Python objects, so the rest of your code can use them.

## 6 · Clear the queue

Now run the same crew on all five tickets. The loop keeps one row per ticket.

Then `print_queue` checks each row against the right answers in `desk_kit.py`:

- **right team:** did the ticket go to the team that owns it?
- **safe reply:** does the reply keep the bank's rules? No refund promises, no deadlines, and it never
  asks for an OTP, PIN or password. Lab 2 works on this column.
- **says what happened:** does the reply tell the customer something from the bank's records? Lab 3
  works on this column. Today the desk can only read the ticket, so expect mostly `NO`.
- **handover:** do all three parts of the handover note have real content?

In [ ]:
import time

async def run_queue(make_crew):
    rows = []
    for ticket_id in TICKETS:
        crew = make_crew()                      # a new crew for each ticket (see section 7)
        out = await crew.kickoff_async(inputs={"ticket_id": ticket_id})
        decision, reply = crew.tasks[1].output.pydantic, out.pydantic
        rows.append({"ticket": ticket_id, "team": decision.team, "priority": decision.priority,
                     "reply": reply.customer_reply, "summary": reply.summary,
                     "what_to_check": reply.what_to_check, "next_action": reply.next_action})
    return rows

before = llm.get_token_usage_summary()
start = time.time()
queue = await run_queue(lambda: Crew(agents=[researcher, classifier, writer], tasks=[t1, t2, t3],
                                     process=Process.sequential, verbose=False))
seconds = time.time() - start
used = llm.get_token_usage_summary().delta_since(before)

print_queue(queue)
print()
print("scorecard:", score(queue))
print(f"five tickets in {seconds:.0f}s, {used.successful_requests} model requests, {used.total_tokens} tokens")

**What the desk did in one run:** every ticket routed, every customer answered, and every team
handed a note it can act on. A person doing this by hand reads each ticket, looks up the team list,
writes two messages and files them.

Look at the rows marked `NO`. Each one is work for a later lab:

- A **wrong team** means that team's queue gets a ticket it cannot solve. Read the team's line in
  `TEAMS` and the ticket. Which words sent it the wrong way?
- A reply that does **not say what happened** can only tell the customer "we are looking into it".
  The desk has only the ticket to read. Lab 3 gives it the bank's records.

The last line is the size of the run. Four requests per ticket: the researcher's tool call and answer,
then one each for the classifier and the writer.

## 7 · Let a manager decide the order

`Process.hierarchical` replaces your task order with a **manager**. The manager is one more model call
that decides which agent does what, and when. It also checks each result.

A manager makes many more model calls, and the number changes from run to run. So this section runs it
on **one ticket only**: `GB-T-4475`, the hardest one in the queue. Compare its answer with the row the
ordinary crew wrote in section 6. Did the manager route it better, or write a better reply?

`managed_desk` builds a new crew each time it is called. A hierarchical crew adds its manager when it
starts. Start the same crew object a second time and CrewAI stops with
`Manager agent should not have tools`. That is also why `run_queue` builds a new crew for each ticket.

In [ ]:
def managed_desk():
    return Crew(
        agents=[researcher, classifier, writer],
        tasks=[t1, t2, t3],
        process=Process.hierarchical,
        manager_llm=llm,
        verbose=False,
    )

crew = managed_desk()
before = llm.get_token_usage_summary()
start = time.time()
out = await crew.kickoff_async(inputs={"ticket_id": "GB-T-4475"})
managed_seconds = time.time() - start
managed_used = llm.get_token_usage_summary().delta_since(before)

ordinary = next(row for row in queue if row["ticket"] == "GB-T-4475")
managed_team = crew.tasks[1].output.pydantic.team if crew.tasks[1].output.pydantic else "?"
managed_reply = out.pydantic.customer_reply if out.pydantic else out.raw

print("IN ORDER     team:", ordinary["team"], "| safe:", check_reply(ordinary["reply"]) or "yes")
print("  ", ordinary["reply"])
print()
print("WITH MANAGER team:", managed_team, "| safe:", check_reply(managed_reply) or "yes")
print("  ", managed_reply)
print()
print(f"in order    : {seconds / len(queue):.0f}s and {used.successful_requests / len(queue):.0f} requests per ticket (average)")
print(f"with manager: {managed_seconds:.0f}s and {managed_used.successful_requests} requests for this one ticket")

### What to take away

- **A crew turns a queue into routed, answered, handed-over work** from a few lines of role and task
  text. `output_pydantic` makes each answer a record your code can use.
- **Check the work against known answers.** Five tickets with a known right team is a small test set.
  It is enough to see whether a change helps.
- **Add a manager only when you can say why.** On this queue the order of the three steps is always the
  same. If the manager did not do the job better on the hardest ticket, it only added requests
  and time. It can also do the job worse: a manager that rewrites the plan can drop the writer's
  rules, and promise the customer a refund.

---

**Next:** Lab 2 makes sure no reply breaks a bank rule before a customer reads it.